# 01 TF-IDF Logistic Regression Baselines

Baseline 1 from the manuscript: unweighted TF-IDF + Logistic Regression with separate word and character TF-IDF branches capped at 5,000 total features.

In [ ]:
from pathlib import Path
import os

# Works from the repository root, from notebooks/, or in Colab after setting BASE_DIR.
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content/thesis-modeling')]
BASE_DIR = next((p for p in CANDIDATES if (p / 'data' / '06_model_ready').exists()), Path('..')).resolve()
DATA_DIR = BASE_DIR / 'data' / '06_model_ready'
RESULTS_DIR = BASE_DIR / 'results'
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
REPORTS_DIR = BASE_DIR / 'reports'
MODELS_DIR = BASE_DIR / 'trained_models'
for d in [RESULTS_DIR, ARTIFACTS_DIR, REPORTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('BASE_DIR =', BASE_DIR)
print('DATA_DIR exists =', DATA_DIR.exists())

import json
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABELS = [0, 1]
TARGET_NAMES = ['ham', 'smishing']

def load_split(path):
    df = pd.read_csv(path)
    if 'model_text' not in df.columns or 'label_id' not in df.columns:
        raise ValueError(f'{path} must contain model_text and label_id')
    df['model_text'] = df['model_text'].fillna('').astype(str)
    df['label_id'] = df['label_id'].astype(int)
    return df

def binary_metrics(y_true, y_pred, model_name, split_name, seed=None):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average='binary', pos_label=1, zero_division=0
    )
    per_class = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average=None, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    tn, fp, fn, tp = cm.ravel()
    row = {
        'model': model_name,
        'split': split_name,
        'seed': seed,
        'n_rows': int(len(y_true)),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_smishing': precision,
        'recall_smishing': recall,
        'f1_smishing': f1,
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_ham': per_class[2][0],
        'false_negative_rate': fn / (fn + tp) if (fn + tp) else 0.0,
        'false_positive_rate': fp / (fp + tn) if (fp + tn) else 0.0,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }
    return row, cm

def save_classification_report(y_true, y_pred, path, title):
    text = classification_report(y_true, y_pred, labels=LABELS, target_names=TARGET_NAMES, zero_division=0)
    path.write_text(f'# {title}\n\n```text\n{text}\n```\n', encoding='utf-8')

from joblib import dump
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression

TFIDF_MODEL_DIR = MODELS_DIR / 'tfidf_logreg'
TFIDF_ARTIFACT_DIR = ARTIFACTS_DIR / 'tfidf_logreg'
TFIDF_MODEL_DIR.mkdir(parents=True, exist_ok=True)
TFIDF_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

train_df = load_split(DATA_DIR / 'clean' / 'train_clean.csv')
test_files = {
    'test_clean': DATA_DIR / 'clean' / 'test_clean.csv',
    'test_adv_10': DATA_DIR / 'adversarial_test' / 'test_adv_10.csv',
    'test_adv_20': DATA_DIR / 'adversarial_test' / 'test_adv_20.csv',
    'test_adv_30': DATA_DIR / 'adversarial_test' / 'test_adv_30.csv',
}
for name, path in test_files.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing expected evaluation file for {name}: {path}')

# Manuscript cap is 5,000 total features. Split the budget across the two TF-IDF branches.
word_vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=2500,
    min_df=2, max_df=0.95, sublinear_tf=True, lowercase=True
)
char_vectorizer = TfidfVectorizer(
    analyzer='char', ngram_range=(1, 3), max_features=2500,
    min_df=2, max_df=0.95, sublinear_tf=True, lowercase=True
)
features = FeatureUnion([('word_tfidf', word_vectorizer), ('char_tfidf', char_vectorizer)])
model = Pipeline([
    ('features', features),
    ('classifier', LogisticRegression(class_weight=None, random_state=SEED, max_iter=1000, solver='lbfgs')),
])

X_train = train_df['model_text']
y_train = train_df['label_id']
model.fit(X_train, y_train)
train_matrix = model.named_steps['features'].transform(X_train)
if train_matrix.shape[1] > 5000:
    raise AssertionError(f'TF-IDF feature cap violated: {train_matrix.shape[1]} > 5000')

metrics = []
for split_name, path in test_files.items():
    df = load_split(path)
    y_true = df['label_id'].to_numpy()
    y_pred = model.predict(df['model_text'])
    row, cm = binary_metrics(y_true, y_pred, 'tfidf_logreg', split_name)
    row['tfidf_total_features'] = int(train_matrix.shape[1])
    metrics.append(row)

    fig, ax = plt.subplots(figsize=(4, 3))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=np.arange(len(TARGET_NAMES)), yticks=np.arange(len(TARGET_NAMES)),
           xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES,
           xlabel='Predicted', ylabel='True',
           title=f'TF-IDF Logistic Regression: {split_name}')
    thresh = cm.max() / 2.0 if cm.max() else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black')
    fig.tight_layout()
    fig.savefig(TFIDF_ARTIFACT_DIR / f'confusion_matrix_{split_name}.png', dpi=200)
    plt.close(fig)

    save_classification_report(
        y_true, y_pred,
        REPORTS_DIR / f'tfidf_logreg_classification_report_{split_name}.md',
        f'TF-IDF Logistic Regression Classification Report - {split_name}'
    )

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(RESULTS_DIR / 'tfidf_logreg_metrics.csv', index=False)
dump(model, TFIDF_MODEL_DIR / 'tfidf_logreg_pipeline.joblib')
metadata = {
    'model': 'Baseline 1: TF-IDF with Logistic Regression, unweighted',
    'class_weight': None,
    'word_max_features': 2500,
    'char_max_features': 2500,
    'tfidf_total_features': int(train_matrix.shape[1]),
    'positive_label': 'smishing / label_id=1',
    'evaluation_files': {k: str(v.relative_to(BASE_DIR)) for k, v in test_files.items()},
}
(TFIDF_MODEL_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(REPORTS_DIR / 'tfidf_logreg_summary.md').write_text(
    '# TF-IDF Logistic Regression Summary\n\n'
    'Baseline 1 uses separate word-level and character-level TF-IDF branches combined with FeatureUnion. '
    'The manuscript max_features=5,000 cap is enforced by allocating 2,500 features to each branch. '
    'Logistic Regression is unweighted with class_weight=None. Primary precision, recall, F1, FNR, and FPR use smishing as the positive class (label_id=1).\n\n'
    + metrics_df.to_string(index=False),
    encoding='utf-8'
)
print(metrics_df)
print('TF-IDF total features:', train_matrix.shape[1])
